# Book Visualizations

Publication-oriented analysis, tables and figures for *Fragrance Accord Intelligence*.

This notebook reproduces the required analytical outputs from the project's source notebooks and prepares them in a consistent format for the book.

## Chapter 1 — Mapping the Fragrance Landscape

### Final Chapter 1 inventory

| Item | Title | Source |
|---|---|---|
| Table 1.1 | Dataset Profile | Dataset profiling |
| Table 1.2 | Availability of Ratings and Accord Data | Dataset profiling |
| Table 1.3 | Fragrance Distribution by Gender Classification | Dataset profiling |
| Table 1.4 | Ten Most Prevalent Fragrance Accords | Notebook 08 |
| Figure 1.1 | Dataset Availability Matrix | Dataset profiling |
| Figure 1.2 | Fragrance Catalogue Distribution by Gender | Dataset profiling |
| Figure 1.3 | Most Prevalent Fragrance Accords | Notebook 08 |

The notebook follows this inventory. Calculations are reproduced for publication and validated before figure production.

**Interpretation conventions**
- A valid accord means a parsed, non-null, non-empty accord label after lower-case and whitespace normalization.
- Accord prevalence uses fragrances with at least one valid accord as its denominator.
- A fragrance may contain multiple accords; prevalence percentages therefore do not sum to 100%.
- Gender labels describe Fragrantica's product classifications, not consumer demographics.
- Ratings and rating counts are platform observations, not sales or direct measures of market demand.


## 0. Setup and source data

Load the cleaned Fragrantica Delta table and reproduce the accord parsing logic from Notebook 08. The exploded dataset contains one row per fragrance–accord combination.

In [ ]:
# ============================================================
# NOTEBOOK 14 — BOOK VISUALIZATIONS
# Chapter 1: Mapping the Fragrance Landscape
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Source table
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
df_perfumes = spark.table(SOURCE_TABLE)

# Shared publication settings — initial book palette
BOOK_STYLE = {
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "background": "#FFFFFF",
    "grid": "#E9E3DD",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["muted"],
    "ytick.color": BOOK_STYLE["muted"],
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Parse main_accords using the same logic as Notebook 08
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'",
                '"'
            ),
            r"\bNone\b",
            "null"
        ),
        "array<string>"
    )
)

# One row per fragrance–accord combination.
# Null and empty accord labels are excluded, as in Notebook 08.
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.lower(F.trim(F.col("accord"))))
    .filter(
        F.col("accord").isNotNull()
        & (F.col("accord") != "")
    )
)

print(f"Source table: {SOURCE_TABLE}")
print(f"Source records: {df_perfumes.count():,}")
print(f"Distinct fragrance IDs: {df_perfumes.select('id').distinct().count():,}")
print(f"Distinct valid accord labels: {df_fragrance_accords.select('accord').distinct().count():,}")

## 1.1 Table 1.1 — Dataset Profile

The profile distinguishes records with a non-null source accord field from records that produce at least one valid parsed accord. These are not interchangeable measures.

In [ ]:
# ============================================================
# TABLE 1.1 — DATASET PROFILE
# ============================================================

total_records = df_perfumes.count()
distinct_ids = df_perfumes.select("id").distinct().count()

valid_accord_ids = (
    df_fragrance_accords
    .select("id")
    .distinct()
)

rated_ids = (
    df_perfumes
    .filter(F.col("rating_value").isNotNull())
    .select("id")
    .distinct()
)

profile = df_perfumes.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("id").alias("distinct_ids"),
    F.countDistinct(
        F.when(F.col("rating_value").isNotNull(), F.col("id"))
    ).alias("records_with_rating"),
    F.countDistinct(
        F.when(F.col("rating_count").isNotNull(), F.col("id"))
    ).alias("records_with_rating_count"),
    F.countDistinct(
        F.when(
            F.col("main_accords").isNotNull(),
            F.col("id")
        )
    ).alias("records_with_non_null_accord_field")
)

valid_accord_count = valid_accord_ids.count()
rated_valid_accord_count = (
    valid_accord_ids
    .join(rated_ids, on="id", how="inner")
    .count()
)

valid_accord_without_rating_count = (
    valid_accord_ids
    .join(rated_ids, on="id", how="left_anti")
    .count()
)

profile_row = profile.first().asDict()

profile_row["records_with_valid_accord_field"] = valid_accord_count
profile_row["rated_records_with_valid_accords"] = rated_valid_accord_count
profile_row["valid_accord_records_without_rating"] = valid_accord_without_rating_count

table_1_1 = pd.DataFrame([
    {"Metric": "Total fragrance records", "Value": profile_row["total_records"]},
    {"Metric": "Distinct fragrance IDs", "Value": profile_row["distinct_ids"]},
    {"Metric": "Records with a rating", "Value": profile_row["records_with_rating"]},
    {"Metric": "Records with a rating count", "Value": profile_row["records_with_rating_count"]},
    {"Metric": "Records with a non-null accord field", "Value": profile_row["records_with_non_null_accord_field"]},
    {"Metric": "Records with at least one valid accord", "Value": profile_row["records_with_valid_accord_field"]},
    {"Metric": "Rated records with at least one valid accord", "Value": profile_row["rated_records_with_valid_accords"]},
    {"Metric": "Valid-accord records without a rating", "Value": profile_row["valid_accord_records_without_rating"]}
])

table_1_1["Value"] = table_1_1["Value"].map(lambda x: f"{x:,}")
display(table_1_1)

## 1.2 Table 1.2 — Availability of Ratings and Accord Data

The four categories below are mutually exclusive. They are calculated from record-level flags rather than inferred by subtracting aggregate totals. The reconciliation assertion verifies that every record belongs to exactly one category.

In [ ]:
# ============================================================
# TABLE 1.2 — AVAILABILITY OF RATINGS AND ACCORD DATA
# ============================================================

valid_accord_flag = valid_accord_ids.withColumn(
    "has_valid_accords",
    F.lit(True)
)

availability = (
    df_perfumes
    .select("id", "rating_value")
    .dropDuplicates(["id"])
    .join(valid_accord_flag, on="id", how="left")
    .withColumn(
        "has_rating",
        F.col("rating_value").isNotNull()
    )
    .withColumn(
        "has_valid_accords",
        F.coalesce(F.col("has_valid_accords"), F.lit(False))
    )
)

availability_counts = availability.agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(F.col("has_rating") & F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("accord_and_rating"),
    F.sum(
        F.when(~F.col("has_rating") & F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("accord_only"),
    F.sum(
        F.when(F.col("has_rating") & ~F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("rating_only"),
    F.sum(
        F.when(~F.col("has_rating") & ~F.col("has_valid_accords"), 1).otherwise(0)
    ).alias("neither")
).first().asDict()

group_total = (
    availability_counts["accord_and_rating"]
    + availability_counts["accord_only"]
    + availability_counts["rating_only"]
    + availability_counts["neither"]
)

assert group_total == availability_counts["total"], (
    "Availability groups do not reconcile to the total record count."
)

table_1_2 = pd.DataFrame([
    {
        "Data availability": "Valid accord data and rating",
        "Fragrance records": availability_counts["accord_and_rating"]
    },
    {
        "Data availability": "Valid accord data, no rating",
        "Fragrance records": availability_counts["accord_only"]
    },
    {
        "Data availability": "Rating, no valid accord data",
        "Fragrance records": availability_counts["rating_only"]
    },
    {
        "Data availability": "Neither rating nor valid accord data",
        "Fragrance records": availability_counts["neither"]
    }
])

table_1_2["Share of dataset"] = (
    table_1_2["Fragrance records"] / availability_counts["total"] * 100
).map(lambda x: f"{x:.2f}%")

table_1_2["Fragrance records"] = (
    table_1_2["Fragrance records"].map(lambda x: f"{x:,}")
)

display(table_1_2)
print(f"Reconciled records: {group_total:,} / {availability_counts['total']:,}")

## 1.3 Table 1.3 — Fragrance Distribution by Gender Classification

Original category values are retained, including `n/a` and null. Shares use the full distinct-ID catalogue as the denominator. Rating coverage is calculated within each classification.

In [ ]:
# ============================================================
# TABLE 1.3 — FRAGRANCE DISTRIBUTION BY GENDER CLASSIFICATION
# ============================================================

gender_source = df_perfumes.withColumn(
    "gender_label",
    F.when(F.col("gender").isNull(), "null")
     .otherwise(F.lower(F.trim(F.col("gender"))))
)

gender_summary = (
    gender_source
    .groupBy("gender_label")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.countDistinct(
            F.when(F.col("rating_value").isNotNull(), F.col("id"))
        ).alias("rated_fragrance_count"),
        F.countDistinct(
            F.when(F.col("main_accords").isNotNull(), F.col("id"))
        ).alias("records_with_non_null_accord_field")
    )
    .withColumn(
        "share_of_catalogue_pct",
        F.round(F.col("fragrance_count") / F.lit(distinct_ids) * 100, 2)
    )
    .withColumn(
        "rating_coverage_pct",
        F.round(
            F.col("rated_fragrance_count") / F.col("fragrance_count") * 100,
            2
        )
    )
    .orderBy(F.desc("fragrance_count"))
)

table_1_3 = gender_summary.toPandas().rename(columns={
    "gender_label": "Gender classification",
    "fragrance_count": "Fragrance records",
    "share_of_catalogue_pct": "Share of catalogue",
    "rated_fragrance_count": "Records with rating",
    "rating_coverage_pct": "Rating coverage",
    "records_with_non_null_accord_field": "Records with non-null accord field"
})

table_1_3["Fragrance records"] = table_1_3["Fragrance records"].map(lambda x: f"{x:,}")
table_1_3["Records with rating"] = table_1_3["Records with rating"].map(lambda x: f"{x:,}")
table_1_3["Records with non-null accord field"] = (
    table_1_3["Records with non-null accord field"].map(lambda x: f"{x:,}")
)
table_1_3["Share of catalogue"] = table_1_3["Share of catalogue"].map(lambda x: f"{x:.2f}%")
table_1_3["Rating coverage"] = table_1_3["Rating coverage"].map(lambda x: f"{x:.2f}%")

display(table_1_3)

## 1.4 Table 1.4 — Ten Most Prevalent Fragrance Accords

This reproduces Notebook 08's prevalence definition:

`Prevalence (%) = fragrance count containing accord / fragrances with at least one valid accord × 100`

The rating metrics are included as descriptive context; Chapter 2 will examine rating patterns in greater detail.

In [ ]:
# ============================================================
# TABLE 1.4 — TEN MOST PREVALENT FRAGRANCE ACCORDS
# ============================================================

fragrance_accords_unique = df_fragrance_accords.dropDuplicates(["id", "accord"])

accord_prevalence = (
    fragrance_accords_unique
    .groupBy("accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.countDistinct(
            F.when(F.col("rating_value").isNotNull(), F.col("id"))
        ).alias("rated_fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("avg_rating")
    )
    .withColumn(
        "prevalence_pct",
        F.round(
            F.col("fragrance_count") / F.lit(valid_accord_count) * 100,
            2
        )
    )
)

top_10_accords = (
    accord_prevalence
    .orderBy(F.desc("fragrance_count"), F.asc("accord"))
    .limit(10)
)

table_1_4 = (
    top_10_accords
    .select(
        F.initcap("accord").alias("Accord"),
        F.col("fragrance_count").alias("Fragrance count"),
        F.col("prevalence_pct").alias("Prevalence (%)"),
        F.col("rated_fragrance_count").alias("Rated fragrance count"),
        F.col("avg_rating").alias("Average rating")
    )
    .toPandas()
)

display(table_1_4)
print(f"Prevalence denominator: {valid_accord_count:,} fragrances")

## 1.5 Chapter 1 — Validation checks

Run these checks before using the outputs in the manuscript. The baseline assertions reflect the current dataset snapshot used to prepare Chapter 1. If the source table is intentionally refreshed, review the changed baseline before updating these values.

In [ ]:
# ============================================================
# CHAPTER 1 — VALIDATION CHECKS
# ============================================================

assert total_records == 69_948, f"Unexpected record count: {total_records:,}"
assert distinct_ids == total_records, "Fragrance IDs are not unique."
assert valid_accord_count == 68_511, (
    f"Unexpected valid-accord count: {valid_accord_count:,}"
)
assert len(table_1_4) == 10, "Top-10 accord table does not contain 10 rows."

distinct_accord_count = accord_prevalence.select("accord").distinct().count()
assert distinct_accord_count == 90, (
    f"Expected 90 accord labels, found {distinct_accord_count}."
)

print("Chapter 1 table validation passed.")
print(f"Records: {total_records:,}")
print(f"Distinct IDs: {distinct_ids:,}")
print(f"Records with valid accords: {valid_accord_count:,}")
print(f"Distinct accord labels: {distinct_accord_count}")
print(f"Availability groups reconciled: {group_total:,}")

## 1.6 Figure 1.1 — Dataset Availability Matrix

The matrix visualizes the same four mutually exclusive groups as Table 1.2. Counts and percentages are displayed within each cell.

In [ ]:
# ============================================================
# FIGURE 1.1 — DATASET AVAILABILITY MATRIX
# ============================================================

from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8.5, 5.5))

matrix = [
    [
        availability_counts["accord_and_rating"],
        availability_counts["accord_only"]
    ],
    [
        availability_counts["rating_only"],
        availability_counts["neither"]
    ]
]

row_labels = ["Valid accord data", "No valid accord data"]
col_labels = ["Rating available", "Rating unavailable"]
cell_colors = [
    [BOOK_STYLE["accent_light"], "#F0EAE4"],
    ["#E7DED5", "#D8D1CA"]
]

for row in range(2):
    for col in range(2):
        value = matrix[row][col]
        pct = value / availability_counts["total"] * 100

        ax.add_patch(Rectangle(
            (col, 1 - row), 1, 1,
            facecolor=cell_colors[row][col],
            edgecolor="white",
            linewidth=3
        ))

        ax.text(
            col + 0.5,
            1.5 - row,
            f"{value:,}\n{pct:.1f}%",
            ha="center",
            va="center",
            fontsize=16,
            fontweight="semibold",
            color=BOOK_STYLE["text"]
        )

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5], col_labels)
ax.set_yticks([1.5, 0.5], row_labels)
ax.tick_params(length=0, labelsize=10)
ax.set_title("Dataset Availability Matrix", loc="left", pad=18)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.text(
    0.12, 0.02,
    f"Total dataset: {availability_counts['total']:,} fragrance records",
    fontsize=9,
    color=BOOK_STYLE["muted"]
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## 1.7 Figure 1.2 — Fragrance Catalogue Distribution by Gender

The three principal Fragrantica classifications are shown as horizontal bars. The number of records outside these three categories is noted separately so that the very small `n/a` and null categories remain visible without distorting the chart scale.

In [ ]:
# ============================================================
# FIGURE 1.2 — FRAGRANCE CATALOGUE DISTRIBUTION BY GENDER
# ============================================================

gender_plot = gender_summary.toPandas()

main_categories = gender_plot[
    gender_plot["gender_label"].isin(["men", "women", "unisex"])
].copy()

other_count = gender_plot[
    ~gender_plot["gender_label"].isin(["men", "women", "unisex"])
]["fragrance_count"].sum()

main_categories = main_categories.sort_values(
    "fragrance_count",
    ascending=True
)

gender_colors = {
    "men": BOOK_STYLE["men"],
    "women": BOOK_STYLE["women"],
    "unisex": BOOK_STYLE["unisex"]
}

fig, ax = plt.subplots(figsize=(8.5, 5))

bars = ax.barh(
    main_categories["gender_label"].str.title(),
    main_categories["fragrance_count"],
    color=[gender_colors[g] for g in main_categories["gender_label"]],
    height=0.58
)

for bar, (_, row) in zip(bars, main_categories.iterrows()):
    value = row["fragrance_count"]
    pct = value / distinct_ids * 100
    ax.text(
        bar.get_width() + 350,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}  |  {pct:.2f}%",
        va="center",
        fontsize=10,
        color=BOOK_STYLE["text"]
    )

ax.set_xlim(0, main_categories["fragrance_count"].max() * 1.32)
ax.set_xlabel("Number of fragrance records")
ax.set_title(
    "Fragrance Catalogue Distribution by Gender",
    loc="left",
    pad=18
)

ax.xaxis.grid(True, color=BOOK_STYLE["grid"], linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BOOK_STYLE["grid"])
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", length=0)

fig.text(
    0.13, 0.02,
    f"Other / unclassified categories: {other_count:,} records",
    fontsize=9,
    color=BOOK_STYLE["muted"]
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## 1.8 Figure 1.3 — Most Prevalent Fragrance Accords

The figure uses the same top-ten ranking as Table 1.4. Each bar represents the number of fragrances containing that accord; labels provide both count and prevalence. The denominator is the number of fragrances with at least one valid accord.

In [ ]:
# ============================================================
# FIGURE 1.3 — MOST PREVALENT FRAGRANCE ACCORDS
# ============================================================

accord_plot = top_10_accords.toPandas().sort_values(
    "fragrance_count",
    ascending=True
)

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.barh(
    accord_plot["accord"].str.title(),
    accord_plot["fragrance_count"],
    color=BOOK_STYLE["accent"],
    height=0.62
)

for bar, (_, row) in zip(bars, accord_plot.iterrows()):
    ax.text(
        bar.get_width() + 300,
        bar.get_y() + bar.get_height() / 2,
        f"{row['fragrance_count']:,}  |  {row['prevalence_pct']:.2f}%",
        va="center",
        fontsize=9,
        color=BOOK_STYLE["text"]
    )

ax.set_xlim(0, accord_plot["fragrance_count"].max() * 1.35)
ax.set_xlabel("Number of fragrances containing the accord")
ax.set_title(
    "Ten Most Prevalent Fragrance Accords",
    loc="left",
    pad=18
)

ax.xaxis.grid(True, color=BOOK_STYLE["grid"], linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BOOK_STYLE["grid"])
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", length=0)

fig.text(
    0.13, 0.02,
    f"Prevalence denominator: {valid_accord_count:,} fragrances with at least one valid accord. "
    "Accords may overlap within a fragrance.",
    fontsize=8,
    color=BOOK_STYLE["muted"]
)

plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()